# EVE Frontier Static Data Visualization

Visualize and explore the static data from the most recent release.

## Setup and Data Loading

In [ ]:
# Verify dependencies are available
# These should be installed via Poetry (see pyproject.toml)
try:
    import requests
    import sqlite3
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import plotly.express as px
    import plotly.graph_objects as go
    print("✓ All required dependencies are available")
except ImportError as e:
    print(f"✗ Missing dependency: {e}")
    print("Run: poetry install")

In [ ]:
# Imports and configuration
import os
from pathlib import Path
from typing import Dict, List, Any, Tuple
from datetime import datetime

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Configuration
DATA_DIR = Path('/workspace/data')
DATA_DIR.mkdir(exist_ok=True)

# Most recent release
LATEST_RELEASE = {
    'tag': 'e6c3',
    'name': 'Era 6, Cycle 3',
    'url': 'https://github.com/Scetrov/evefrontier_datasets/releases/download/e6c3/static_data.db',
    'path': DATA_DIR / 'static_data_latest.db'
}

print(f"📊 Visualization Setup Complete")
print(f"📁 Data directory: {DATA_DIR}")
print(f"🎯 Latest release: {LATEST_RELEASE['name']} ({LATEST_RELEASE['tag']})")

In [ ]:
def download_file(url: str, destination: Path, force: bool = False) -> bool:
    """Download a file from URL to destination."""
    if destination.exists() and not force:
        print(f"✓ File already exists: {destination.name}")
        return True
    
    try:
        print(f"⬇️  Downloading {destination.name}...")
        response = requests.get(url, stream=True, timeout=30)
        response.raise_for_status()
        
        total_size = int(response.headers.get('content-length', 0))
        downloaded = 0
        
        with open(destination, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total_size:
                        percent = (downloaded / total_size) * 100
                        print(f"  Progress: {percent:.1f}% ({downloaded / 1024 / 1024:.1f} MB)", end='\r')
        
        print(f"\n✓ Downloaded: {destination.name} ({destination.stat().st_size / 1024 / 1024:.2f} MB)")
        return True
    except Exception as e:
        print(f"✗ Failed to download {destination.name}: {e}")
        return False

# Download latest dataset
download_file(LATEST_RELEASE['url'], LATEST_RELEASE['path'])

In [ ]:
def load_database_tables(db_path: Path) -> Dict[str, pd.DataFrame]:
    """Load all tables from database into DataFrames."""
    conn = sqlite3.connect(db_path)
    
    # Get all table names
    cursor = conn.cursor()
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
    tables = [row[0] for row in cursor.fetchall()]
    
    # Load each table
    data = {}
    for table in tables:
        try:
            data[table] = pd.read_sql_query(f"SELECT * FROM {table}", conn)
        except Exception as e:
            print(f"⚠️  Error loading table {table}: {e}")
    
    conn.close()
    return data

# Load all tables
print(f"📂 Loading tables from {LATEST_RELEASE['name']}...")
db_data = load_database_tables(LATEST_RELEASE['path'])
print(f"✓ Loaded {len(db_data)} tables")

# Display table overview
print("\n📋 Database Tables:")
for table_name, df in db_data.items():
    print(f"   {table_name:30} | {len(df):>8,} rows | {len(df.columns):>3} columns")

## Database Overview

In [ ]:
# Create overview statistics
overview_data = []
for table_name, df in db_data.items():
    overview_data.append({
        'Table': table_name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Memory (MB)': df.memory_usage(deep=True).sum() / 1024 / 1024
    })

overview_df = pd.DataFrame(overview_data).sort_values('Rows', ascending=False)
print("\n📊 Database Overview")
print(overview_df.to_string(index=False))
print(f"\nTotal Rows: {overview_df['Rows'].sum():,}")
print(f"Total Memory: {overview_df['Memory (MB)'].sum():.2f} MB")

In [ ]:
# Scan for coordinate columns (centerX, centerY, centerZ)
print("\n🔍 Scanning for coordinate columns (centerX, centerY, centerZ)...\n")

coordinate_tables = {}
for table_name, df in db_data.items():
    cols = set(df.columns)
    
    # Look for coordinate columns
    has_x = 'centerX' in cols or 'fromCenterX' in cols or 'toCenterX' in cols
    has_y = 'centerY' in cols or 'fromCenterY' in cols or 'toCenterY' in cols
    has_z = 'centerZ' in cols or 'fromCenterZ' in cols or 'toCenterZ' in cols
    
    if has_x and has_y:
        # Determine which coordinate columns to use
        if 'centerX' in cols:
            x_col, y_col, z_col = 'centerX', 'centerY', 'centerZ' if 'centerZ' in cols else None
        elif 'fromCenterX' in cols:
            x_col, y_col, z_col = 'fromCenterX', 'fromCenterY', 'fromCenterZ' if 'fromCenterZ' in cols else None
        elif 'toCenterX' in cols:
            x_col, y_col, z_col = 'toCenterX', 'toCenterY', 'toCenterZ' if 'toCenterZ' in cols else None
        else:
            continue
        
        coordinate_tables[table_name] = {
            'x_col': x_col,
            'y_col': y_col,
            'z_col': z_col,
            'has_z': z_col is not None
        }

if coordinate_tables:
    print(f"✓ Found {len(coordinate_tables)} tables with coordinate data:\n")
    for table_name, coords in coordinate_tables.items():
        z_info = f"+ Z ({coords['z_col']})" if coords['z_col'] else "2D only"
        print(f"   {table_name:30} | X: {coords['x_col']:15} Y: {coords['y_col']:15} {z_info}")
else:
    print("✗ No tables with coordinate columns found")

In [ ]:
# Dump all column names to see what we're working with
print("\n📋 ALL COLUMN NAMES IN DATABASE\n")
print("="*80)

for table_name in sorted(db_data.keys()):
    df = db_data[table_name]
    print(f"\n{table_name} ({len(df.columns)} columns):")
    print("-" * 80)
    
    # Print columns in rows of 3
    cols = list(df.columns)
    for i in range(0, len(cols), 3):
        row_cols = cols[i:i+3]
        print("  " + " | ".join(f"{col:30}" for col in row_cols))

print("\n" + "="*80)

## Spatial Data Visualization (2D & 3D)

In [ ]:
def plot_2d_scatter(table_name: str, x_col: str, y_col: str, color_col: str = None, size: int = 50):
    """Create a 2D scatter plot from coordinate data."""
    if table_name not in db_data:
        print(f"✗ Table '{table_name}' not found")
        return
    
    df = db_data[table_name]
    
    # Remove rows with NaN coordinates
    plot_df = df.dropna(subset=[x_col, y_col])
    
    if len(plot_df) == 0:
        print(f"✗ No valid coordinate data in {table_name}")
        return
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    if color_col and color_col in df.columns:
        scatter = ax.scatter(plot_df[x_col], plot_df[y_col], 
                            c=pd.to_numeric(plot_df[color_col], errors='coerce'),
                            s=size, alpha=0.6, cmap='viridis', edgecolors='black', linewidth=0.5)
        cbar = plt.colorbar(scatter, ax=ax)
        cbar.set_label(color_col, rotation=270, labelpad=20)
    else:
        ax.scatter(plot_df[x_col], plot_df[y_col], s=size, alpha=0.6, edgecolors='black', linewidth=0.5)
    
    ax.set_xlabel(x_col, fontsize=12)
    ax.set_ylabel(y_col, fontsize=12)
    ax.set_title(f'{table_name} - 2D Spatial Distribution\n({len(plot_df):,} points)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"✓ Plotted {len(plot_df):,} points from {table_name}")

def plot_3d_scatter(table_name: str, x_col: str, y_col: str, z_col: str, color_col: str = None, size: int = 30):
    """Create a 3D scatter plot from coordinate data."""
    if table_name not in db_data:
        print(f"✗ Table '{table_name}' not found")
        return
    
    df = db_data[table_name]
    
    # Remove rows with NaN coordinates
    plot_df = df.dropna(subset=[x_col, y_col, z_col])
    
    if len(plot_df) == 0:
        print(f"✗ No valid 3D coordinate data in {table_name}")
        return
    
    # Use plotly for interactive 3D visualization
    if color_col and color_col in df.columns:
        color_data = pd.to_numeric(plot_df[color_col], errors='coerce')
        fig = px.scatter_3d(
            plot_df,
            x=x_col, y=y_col, z=z_col,
            color=color_col,
            title=f'{table_name} - 3D Spatial Distribution ({len(plot_df):,} points)',
            labels={x_col: x_col, y_col: y_col, z_col: z_col},
            hover_data=list(plot_df.columns[:min(5, len(plot_df.columns))])
        )
    else:
        fig = px.scatter_3d(
            plot_df,
            x=x_col, y=y_col, z=z_col,
            title=f'{table_name} - 3D Spatial Distribution ({len(plot_df):,} points)',
            labels={x_col: x_col, y_col: y_col, z_col: z_col},
            hover_data=list(plot_df.columns[:min(5, len(plot_df.columns))])
        )
    
    fig.update_traces(marker=dict(size=size, opacity=0.7))
    fig.show()
    
    print(f"✓ Plotted {len(plot_df):,} points in 3D space")

# Display helper functions info
print("✓ Plotting functions ready")
print("\nUsage:")
print("  plot_2d_scatter('table_name', 'x_column', 'y_column', color_col='optional_column')")
print("  plot_3d_scatter('table_name', 'x_column', 'y_column', 'z_column', color_col='optional_column')")
print("\nTables with coordinate data:")
if coordinate_tables:
    for table_name in coordinate_tables.keys():
        print(f"  - {table_name}")

In [ ]:
# Generate plots for tables with coordinate data
if coordinate_tables:
    print("\n📊 Generating spatial visualizations...\n")
    
    for table_name, coords in coordinate_tables.items():
        print(f"Plotting {table_name}...")
        
        # 2D plot
        print(f"  → 2D scatter plot ({coords['x_col']} vs {coords['y_col']})")
        plot_2d_scatter(table_name, coords['x_col'], coords['y_col'], size=40)
        
        # 3D plot if Z coordinate exists
        if coords['z_col']:
            print(f"  → 3D scatter plot ({coords['x_col']}, {coords['y_col']}, {coords['z_col']})")
            plot_3d_scatter(table_name, coords['x_col'], coords['y_col'], coords['z_col'], size=5)
        
        print()
else:
    print("⚠️  No coordinate data found in database")

## Interactive Coordinate Explorer

In [ ]:
# Interactive tool to explore any table with coordinates
def explore_coordinates(table_name: str, x_col: str = None, y_col: str = None, z_col: str = None):
    """Interactively explore coordinate data from any table."""
    if table_name not in db_data:
        print(f"✗ Table '{table_name}' not found")
        return
    
    df = db_data[table_name]
    
    # Auto-detect columns if not provided
    if not x_col or not y_col:
        cols = set(df.columns)
        if 'centerX' in cols:
            x_col = x_col or 'centerX'
            y_col = y_col or 'centerY'
            z_col = z_col or ('centerZ' if 'centerZ' in cols else None)
        elif 'fromCenterX' in cols:
            x_col = x_col or 'fromCenterX'
            y_col = y_col or 'fromCenterY'
            z_col = z_col or ('fromCenterZ' if 'fromCenterZ' in cols else None)
        elif 'toCenterX' in cols:
            x_col = x_col or 'toCenterX'
            y_col = y_col or 'toCenterY'
            z_col = z_col or ('toCenterZ' if 'toCenterZ' in cols else None)
    
    if not x_col or not y_col:
        print(f"✗ Could not find X and Y columns in {table_name}")
        print(f"Available columns: {list(df.columns)}")
        return
    
    print(f"\n📍 Exploring {table_name}")
    print(f"   X: {x_col}")
    print(f"   Y: {y_col}")
    if z_col:
        print(f"   Z: {z_col}")
    
    # Statistics
    df_clean = df.dropna(subset=[x_col, y_col])
    print(f"\n📊 Statistics:")
    print(f"   Valid points: {len(df_clean):,}")
    print(f"   X range: [{df_clean[x_col].min():.2e}, {df_clean[x_col].max():.2e}]")
    print(f"   Y range: [{df_clean[y_col].min():.2e}, {df_clean[y_col].max():.2e}]")
    if z_col and z_col in df.columns:
        df_clean_z = df_clean.dropna(subset=[z_col])
        print(f"   Z range: [{df_clean_z[z_col].min():.2e}, {df_clean_z[z_col].max():.2e}]")
    
    # Find numeric columns for coloring
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"\n🎨 Available numeric columns for coloring:")
    for i, col in enumerate(numeric_cols[:10], 1):
        print(f"   {i}. {col}")
    
    return {
        'table': table_name,
        'x_col': x_col,
        'y_col': y_col,
        'z_col': z_col,
        'numeric_cols': numeric_cols
    }

# Example: Explore all tables with coordinates
print("\n📊 Exploring all coordinate tables:\n")
explored_tables = {}
for table_name in coordinate_tables.keys():
    coords = coordinate_tables[table_name]
    explored_tables[table_name] = explore_coordinates(table_name, coords['x_col'], coords['y_col'], coords['z_col'])
    print()

## Solar System Visualization

In [ ]:
def visualize_solar_system(system_id: int, system_name: str = None, size_scale: float = 1.0):
    """
    Visualize a specific solar system with all its celestial objects.
    
    Objects are color-coded:
    - Red: Primary Star (sun at center)
    - Blue: Planets
    - Green: Moons
    - Orange: NPC Stations
    
    Uses logarithmic scaling to handle the huge coordinate ranges.
    
    Args:
        system_id: The solarSystemId to visualize
        system_name: Optional name to display (will look up if not provided)
        size_scale: Scale factor for object sizes (default 1.0)
    """
    
    # Get system info
    if 'SolarSystems' not in db_data:
        print("✗ SolarSystems table not found")
        return
    
    systems_df = db_data['SolarSystems']
    system_info = systems_df[systems_df['solarSystemId'] == system_id]
    
    if len(system_info) == 0:
        print(f"✗ Solar system {system_id} not found")
        return
    
    if system_name is None:
        system_name = system_info.iloc[0]['name']
    
    # Get sun position (center of system)
    sun_x = system_info.iloc[0]['centerX']
    sun_y = system_info.iloc[0]['centerY']
    sun_z = system_info.iloc[0]['centerZ']
    
    print(f"\n⭐ Visualizing Solar System: {system_name} (ID: {system_id})")
    print(f"   Sun position: ({sun_x:.2e}, {sun_y:.2e}, {sun_z:.2e})")
    
    # Prepare data for plotting
    plot_data = {
        'x': [],
        'y': [],
        'z': [],
        'type': [],
        'name': [],
        'size': [],
        'color': []
    }
    
    # Add sun at center
    plot_data['x'].append(sun_x)
    plot_data['y'].append(sun_y)
    plot_data['z'].append(sun_z)
    plot_data['type'].append('Star')
    plot_data['name'].append(f'{system_name} Star')
    plot_data['size'].append(20)
    plot_data['color'].append('Red')
    
    # Add planets
    if 'Planets' in db_data:
        planets_df = db_data['Planets']
        planets = planets_df[planets_df['solarSystemId'] == system_id]
        
        for idx, planet in planets.iterrows():
            plot_data['x'].append(planet['centerX'])
            plot_data['y'].append(planet['centerY'])
            plot_data['z'].append(planet['centerZ'])
            plot_data['type'].append('Planet')
            plot_data['name'].append(planet.get('name', f"Planet {idx}"))
            plot_data['size'].append(12)
            plot_data['color'].append('Blue')
        
        print(f"   ✓ Found {len(planets)} planets")
    
    # Add moons
    if 'Moons' in db_data:
        moons_df = db_data['Moons']
        moons = moons_df[moons_df['solarSystemId'] == system_id]
        
        for idx, moon in moons.iterrows():
            plot_data['x'].append(moon['centerX'])
            plot_data['y'].append(moon['centerY'])
            plot_data['z'].append(moon['centerZ'])
            plot_data['type'].append('Moon')
            plot_data['name'].append(moon.get('name', f"Moon {idx}"))
            plot_data['size'].append(8)
            plot_data['color'].append('Green')
        
        print(f"   ✓ Found {len(moons)} moons")
    
    # Add NPC stations
    if 'NpcStations' in db_data:
        stations_df = db_data['NpcStations']
        stations = stations_df[stations_df['solarSystemId'] == system_id]
        
        for idx, station in stations.iterrows():
            plot_data['x'].append(station['centerX'])
            plot_data['y'].append(station['centerY'])
            plot_data['z'].append(station['centerZ'])
            plot_data['type'].append('NPC Station')
            plot_data['name'].append(station.get('name', f"Station {idx}"))
            plot_data['size'].append(10)
            plot_data['color'].append('Orange')
        
        print(f"   ✓ Found {len(stations)} NPC stations")
    
    # Create interactive 3D scatter plot with plotly
    plot_df = pd.DataFrame(plot_data)
    
    # Center coordinates around the sun (first object, which is the star)
    sun_coords = plot_df.iloc[0][['x', 'y', 'z']]
    plot_df['x_centered'] = plot_df['x'] - sun_coords['x']
    plot_df['y_centered'] = plot_df['y'] - sun_coords['y']
    plot_df['z_centered'] = plot_df['z'] - sun_coords['z']
    
    # Calculate ranges to understand the scale
    x_range = plot_df['x_centered'].max() - plot_df['x_centered'].min()
    y_range = plot_df['y_centered'].max() - plot_df['y_centered'].min()
    z_range = plot_df['z_centered'].max() - plot_df['z_centered'].min()
    
    
    # Find the minimum non-zero distance to use as appropriate offset
    non_zero_distances = []
    for coord in ['x_centered', 'y_centered', 'z_centered']:
        non_zero = plot_df[coord][plot_df[coord] != 0].abs()
        if len(non_zero) > 0:
            non_zero_distances.extend(non_zero.tolist())
    
    # Use 1% of minimum non-zero distance as offset, or 1.0 as fallback
    offset = min(non_zero_distances) * 0.01 if non_zero_distances else 1.0
    
    print(f"   Coordinate ranges:")
    print(f"     X: {plot_df['x_centered'].min():.2e} to {plot_df['x_centered'].max():.2e} (range: {x_range:.2e})")
    print(f"     Y: {plot_df['y_centered'].min():.2e} to {plot_df['y_centered'].max():.2e} (range: {y_range:.2e})")
    print(f"     Z: {plot_df['z_centered'].min():.2e} to {plot_df['z_centered'].max():.2e} (range: {z_range:.2e})")
    print(f"     Log offset: {offset:.2e}")
    
    # Use logarithmic scaling to handle huge coordinate ranges
    # This prevents precision loss and ensures visibility of all objects
    plot_df['x_scaled'] = np.sign(plot_df['x_centered']) * np.log10(np.abs(plot_df['x_centered']) + offset)
    plot_df['y_scaled'] = np.sign(plot_df['y_centered']) * np.log10(np.abs(plot_df['y_centered']) + offset)
    plot_df['z_scaled'] = np.sign(plot_df['z_centered']) * np.log10(np.abs(plot_df['z_centered']) + offset)
    
    # Color mapping
    color_map = {
        'Red': '#FF0000',
        'Blue': '#0000FF',
        'Green': '#00FF00',
        'Orange': '#FFA500'
    }
    
    fig = go.Figure()
    
    # Add traces for each object type
    for obj_type in plot_df['type'].unique():
        type_data = plot_df[plot_df['type'] == obj_type]
        color = type_data['color'].iloc[0]
        
        fig.add_trace(go.Scatter3d(
            x=type_data['x_scaled'],
            y=type_data['y_scaled'],
            z=type_data['z_scaled'],
            mode='markers',
            name=obj_type,
            marker=dict(
                size=type_data['size'] * size_scale,
                color=color_map[color],
                opacity=0.8,
                line=dict(width=1, color='white')
            ),
            text=[f"<b>{name}</b><br>Type: {t}<br>Relative X: {x:.2e}<br>Relative Y: {y:.2e}<br>Relative Z: {z:.2e}" 
                  for name, t, x, y, z in zip(type_data['name'], type_data['type'], 
                                               type_data['x_centered'], type_data['y_centered'], type_data['z_centered'])],
            hovertemplate='%{text}<extra></extra>'
        ))
    
    # Calculate bounding box for scaling (using log-scaled coordinates)
    x_min, x_max = plot_df['x_scaled'].min(), plot_df['x_scaled'].max()
    y_min, y_max = plot_df['y_scaled'].min(), plot_df['y_scaled'].max()
    z_min, z_max = plot_df['z_scaled'].min(), plot_df['z_scaled'].max()
    
    # Add padding (15% of range)
    x_range_scaled = x_max - x_min
    y_range_scaled = y_max - y_min
    z_range_scaled = z_max - z_min
    
    x_pad = x_range_scaled * 0.15 if x_range_scaled > 0 else 1
    y_pad = y_range_scaled * 0.15 if y_range_scaled > 0 else 1
    z_pad = z_range_scaled * 0.15 if z_range_scaled > 0 else 1
    
    fig.update_layout(
        title=f'<b>{system_name}</b><br>Solar System Overview (Log-Scaled)',
        scene=dict(
            xaxis=dict(
                range=[x_min - x_pad, x_max + x_pad],
                title='Log10(|Relative X|) Coordinate'
            ),
            yaxis=dict(
                range=[y_min - y_pad, y_max + y_pad],
                title='Log10(|Relative Y|) Coordinate'
            ),
            zaxis=dict(
                range=[z_min - z_pad, z_max + z_pad],
                title='Log10(|Relative Z|) Coordinate'
            ),
            aspectmode='auto',
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=1.5)
            )
        ),
        width=1200,
        height=800,
        showlegend=True,
        hovermode='closest'
    )
    
    fig.show()
    
    print(f"   📊 Total objects: {len(plot_df)}")
    print(f"   ✅ Visualization complete")
    
    return plot_df

# List available solar systems
print("\n🌍 Available Solar Systems:\n")
if 'SolarSystems' in db_data:
    systems_df = db_data['SolarSystems']
    print(f"Total systems: {len(systems_df)}\n")
    print("Sample systems:")
    for idx, (_, row) in enumerate(systems_df.head(10).iterrows()):
        print(f"   {row['solarSystemId']:>8} | {row['name']}")
    
    print("\n📌 Usage:")
    print("   visualize_solar_system(system_id, system_name='Optional Name', size_scale=1.0)")
    print("\n   Example:")
    print(f"   visualize_solar_system({systems_df.iloc[0]['solarSystemId']}, '{systems_df.iloc[0]['name']}')")

In [ ]:
# Interactive Solar System Query
if 'SolarSystems' in db_data:
    systems_df = db_data['SolarSystems']
    
    # Function to search for solar systems
    def search_solar_system(query: str):
        """Search for solar systems by name (case-insensitive)."""
        results = systems_df[systems_df['name'].str.contains(query, case=False, na=False)]
        
        if len(results) == 0:
            print(f"✗ No solar systems found matching '{query}'")
            return None
        
        print(f"\n🔍 Found {len(results)} matching system(s):\n")
        for idx, (_, row) in enumerate(results.iterrows(), 1):
            print(f"   {idx}. {row['name']:30} (ID: {row['solarSystemId']})")
        
        return results
    
    # Example: Search for a solar system
    print("🔎 Solar System Search Tool")
    print("\nUsage: search_solar_system('system_name')")
    print("Examples:")
    print("  search_solar_system('I4F-MCH')")
    print("  search_solar_system('Strym')")
    print("  search_solar_system('Brana')")
    
    # Perform a search (you can modify this query)
    search_query = 'Brana'  # Change this to search for different systems
    
    print(f"\n{'='*60}")
    search_results = search_solar_system(search_query)
    
    if search_results is not None and len(search_results) > 0:
        # Visualize the first matching system
        selected_system = search_results.iloc[0]
        system_id = selected_system['solarSystemId']
        system_name = selected_system['name']
        
        print(f"\n{'='*60}")
        print(f"✓ Visualizing: {system_name}\n")
        system_plot_data = visualize_solar_system(system_id, system_name)


## Find System with Many Planets

In [ ]:
# Find and visualize a system with many planets
print("\n🌍 Finding system with most planets...\n")

if 'Planets' in db_data and 'SolarSystems' in db_data:
    planets_df = db_data['Planets']
    systems_df = db_data['SolarSystems']
    
    # Count planets per system
    planet_counts = planets_df['solarSystemId'].value_counts()
    
    if len(planet_counts) > 0:
        # Get the system with the most planets
        best_system_id = planet_counts.idxmax()
        planet_count = planet_counts.iloc[0]
        
        # Get system info
        system_info = systems_df[systems_df['solarSystemId'] == best_system_id]
        if len(system_info) > 0:
            system_name = system_info.iloc[0]['name']
            
            print(f"✨ System with most planets: {system_name}")
            print(f"   ID: {best_system_id}")
            print(f"   Planets: {planet_count}")
            
            # Get additional stats
            moons_count = len(db_data['Moons'][db_data['Moons']['solarSystemId'] == best_system_id]) if 'Moons' in db_data else 0
            stations_count = len(db_data['NpcStations'][db_data['NpcStations']['solarSystemId'] == best_system_id]) if 'NpcStations' in db_data else 0
            
            print(f"   Moons: {moons_count}")
            print(f"   NPC Stations: {stations_count}")
            print(f"   Total objects: {planet_count + moons_count + stations_count + 1} (including star)")
            
            print(f"\n📊 Visualizing {system_name}...\n")
            visualize_solar_system(best_system_id, system_name)
    else:
        print("✗ No planets found in database")
else:
    print("✗ Required tables not found")